# Flujo de auditoría

Aquí se muestra información de los datos en cada una de las 3 capas del pipeline. La información se concentra en tablas Parquet ubicadas junto a los datos de cada capa.
Todos los registros de auditoría tienen un UUID único por ejecución del pipeline y referencian al UUID de la ejecución de la capa anterior sobre la que trabajan.

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

from pyspark.sql import functions as F
from app.utils.spark import SparkClient
from app.utils.globals import globals

spark_client = SparkClient()
spark = spark_client.get_session()

BRONZE_PATH = "data/bronze"
SILVER_PATH = "data/silver"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 09:20:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/28 09:20:08 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/28 09:20:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Capa bronce

La capa bronce contempla los datos en bruto provenientes de NYC TLC.

Información almacenada:

* UUID
* Nombre de cada dataset
* Ruta en disco del archivo descargado
* Tamaño en bytes
* Número de filas
* Fecha de inicio y fin de la descarga


In [3]:
df = spark.read.parquet(str(globals.project_root / BRONZE_PATH))
df.show()

+--------------------+-----------------+--------------------+---------+--------+--------------------+--------------------+
|            audit_id|             name|         source_file|bytecount|rowcount|     start_timestamp|       end_timestamp|
+--------------------+-----------------+--------------------+---------+--------+--------------------+--------------------+
|85befb26-a78d-409...|zone-lookup-table|data/bronze/zone-...|     4968|     265|2026-06-26T18:56:...|2026-06-26T18:56:...|
|85befb26-a78d-409...|    fhvhv_2025-01|data/bronze/fhvhv...|491076642|20405666|2026-06-26T18:56:...|2026-06-26T18:57:...|
|85befb26-a78d-409...|    fhvhv_2025-02|data/bronze/fhvhv...|461632254|19339461|2026-06-26T18:57:...|2026-06-26T18:58:...|
|85befb26-a78d-409...|    fhvhv_2025-03|data/bronze/fhvhv...|501783767|20536879|2026-06-26T18:58:...|2026-06-26T18:59:...|
|85befb26-a78d-409...|    fhvhv_2025-04|data/bronze/fhvhv...|487219586|19753983|2026-06-26T18:59:...|2026-06-26T18:59:...|
|85befb26-a78d-4

### Número total de registros

Muestra el número total de registros por ejecución

In [10]:
num = df.orderBy(df.start_timestamp.desc()).groupBy("audit_id").agg(F.sum("rowcount"))
num.show()

+--------------------+-------------+
|            audit_id|sum(rowcount)|
+--------------------+-------------+
|12a61a8d-22f5-43f...|    904328127|
|85befb26-a78d-409...|    243589949|
+--------------------+-------------+



## Capa plata

En la capa de plata se realiza la limpieza y estandarización de los datos, que es el objetivo principal de la auditoría. Con el objetivo de mantener la trazabilidad, los registros que se consideran "sucios" son colocados en cuarentena, con el objetivo de permitir su examinación. 

Información almacenada:

* UUID
* UUID de la ejecución de la capa bronze
* Fecha de inicio y fin del procesamiento
* Ruta en disco del archivo procesado
* Número de filas original
* Número de filas aceptadas
* Número de filas en cuarentena

In [11]:
df = spark.read.parquet(str(globals.project_root / SILVER_PATH))
df.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+----------------+--------------------+
|            audit_id|     bronze_audit_id|     start_timestamp|       end_timestamp|         source_file|rowcount_bronze|rowcount_quality|rowcount_quarantined|
+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+----------------+--------------------+
|628c8d48-9dbd-43a...|12a61a8d-22f5-43f...|2026-06-28T08:56:...|2026-06-28T08:58:...|data/bronze/fhvhv...|       20405666|        20405529|                 137|
|628c8d48-9dbd-43a...|12a61a8d-22f5-43f...|2026-06-28T08:59:...|2026-06-28T09:00:...|data/bronze/fhvhv...|       19339461|        19339333|                 128|
|628c8d48-9dbd-43a...|12a61a8d-22f5-43f...|2026-06-28T09:01:...|2026-06-28T09:03:...|data/bronze/fhvhv...|       20536879|        20536752|                 127|
|628c8d48-9dbd-43a...|12a61a8d-22f

### Resumen

In [12]:
num = df.orderBy(df.start_timestamp.desc()).groupBy("audit_id").agg(F.sum("rowcount_bronze"), F.sum("rowcount_quality"), F.sum("rowcount_quarantined"))
num.show()

+--------------------+--------------------+---------------------+-------------------------+
|            audit_id|sum(rowcount_bronze)|sum(rowcount_quality)|sum(rowcount_quarantined)|
+--------------------+--------------------+---------------------+-------------------------+
|628c8d48-9dbd-43a...|           243589684|            243575961|                    13723|
|ed973aaa-3db2-463...|            74361521|             73481290|                   880231|
+--------------------+--------------------+---------------------+-------------------------+

